# Lab 03 - Notebooks Avanzados

**Objetivos:**
- Dominar magic commands
- Usar widgets para parametrización
- Comunicación entre notebooks
- Debugging y logging

## Ejercicio 1: Magic Commands

### Markdown

In [ ]:
# =============================================================================
# MAGIC COMMAND: %md (Markdown)
# =============================================================================
# Objetivo: Renderizar markdown directamente sin crear celda separada
# Útil para: Documentación dinámica generada desde código

# %md es un "magic command" que cambia el comportamiento de la celda

# Todo el contenido después de %md se interpreta como Markdown# - %run: Ejecutar otro notebook

# - %pip: Instalar paquetes Python en runtime

%md# - %fs: Comandos del file system (atajos para dbutils.fs)

# Título usando Magic Command# - %sh: Ejecutar comandos shell en el driver node

Este es un **markdown** usando `%md`# - %python, %scala, %r: Cambiar lenguaje de la celda

# - %sql: Ejecutar consultas SQL

- Item 1# - %md: Renderizar Markdown

- Item 2# 💡 MAGIC COMMANDS DISPONIBLES EN DATABRICKS:

- Item 3

### Crear datos de prueba

In [ ]:
# =============================================================================
# GENERACIÓN DE DATOS SINTÉTICOS DE VENTAS
# =============================================================================
# Objetivo: Crear dataset realista para testear widgets y queries
# Sin dependencias externas (todo en memoria)

# Importaciones
from pyspark.sql.functions import *  # Funciones de transformación
from datetime import datetime, timedelta  # Manejo de fechas
import random  # Generación aleatoria

# Datos maestros (catálogos)
products = ["Laptop", "Mouse", "Keyboard", "Monitor", "Webcam"]
regions = ["North", "South", "East", "West"]

# Generar 100 ventas aleatorias
data = []
for i in range(100):
    data.append({
        # ID de venta con formato S0001, S0002, etc.
        "sale_id": f"S{i:04d}",  # :04d = 4 dígitos con padding de ceros
        
        # Fecha aleatoria en los últimos 30 días
        "date": (datetime.now() - timedelta(days=random.randint(0, 30))).strftime("%Y-%m-%d"),
        

        # Producto aleatorio del catálogo# - Prototipado rápido de pipelines

        "product": random.choice(products),# - Unit tests de lógica de negocio

        # - Demos y capacitaciones

        # Región aleatoria# - Desarrollo y testing (no requiere datos reales)

        "region": random.choice(regions),# 💡 NOTA: Este enfoque es ideal para:

        

        # Monto entre $100 - $2000, con 2 decimalesdisplay(df_sales.limit(5))  # Mostrar solo 5 filas de ejemplo

        "amount": round(random.uniform(100, 2000), 2),print(f"✅ Datos creados: {df_sales.count()} ventas")

        

        # Cantidad entre 1-5 unidadesdf_sales.createOrReplaceTempView("sales")

        "quantity": random.randint(1, 5)# "sales" será el nombre de la tabla en queries SQL

    })# Crear vista temporal para poder usar SQL



# Crear DataFrame desde lista de diccionariosdf_sales = spark.createDataFrame(data)
# Spark infiere el schema automáticamente desde las keys/values

### SQL Query

In [ ]:
# =============================================================================
# MAGIC COMMAND: %%sql (SQL Query)
# =============================================================================
# Objetivo: Ejecutar SQL directamente contra temp views
# Ventaja: No necesitas spark.sql("..."), más limpio y legible

# %%sql (con doble %%) convierte TODA la celda a SQL
# Puedes usar TODA la sintaxis de SQL estándar

%%sql
-- Consulta: Top 10 combinaciones Region-Product por ingresos

-- Usamos agregaciones (COUNT, SUM, AVG) y ordenamiento-- Databricks automáticamente renderiza gráficos (barras, líneas, pie, etc.)

-- 📊 RESULTADO: Tabla interactiva con opciones de visualización

SELECT 

    region,                               -- Región de venta--   .limit(10)

    product,                              -- Producto vendido--   .orderBy(col("total_revenue").desc()) \

    COUNT(*) as num_sales,                -- Total de ventas (transacciones)--   ) \

    ROUND(SUM(amount), 2) as total_revenue,  -- Ingresos totales con 2 decimales--     round(avg("amount"), 2).alias("avg_amount")

    ROUND(AVG(amount), 2) as avg_amount   -- Ticket promedio--     round(sum("amount"), 2).alias("total_revenue"),

FROM sales                                -- Desde la temp view "sales"--     count("*").alias("num_sales"),

GROUP BY region, product                  -- Agrupar por región y producto--   .agg(

ORDER BY total_revenue DESC               -- Ordenar por ingresos (mayor a menor)-- df_sales.groupBy("region", "product") \

LIMIT 10                                  -- Solo top 10 resultados-- 💡 EQUIVALENTE EN DATAFRAME API:


### Shell Commands

In [ ]:
%sh
echo "📁 Listar directorio actual:"
pwd
ls -lh

### File System Commands

In [ ]:
%fs ls /tmp/

## Ejercicio 2: Widgets (Parametrización)

In [ ]:
# =============================================================================
# WIDGETS: PARAMETRIZACIÓN DE NOTEBOOKS
# =============================================================================
# Objetivo: Hacer notebooks interactivos y reutilizables
# Widgets permiten: Ejecutar mismo notebook con diferentes parámetros

# dbutils.widgets es la API de Databricks para crear controles de entrada

# Los widgets aparecen en la parte superior del notebook# 4. Orchestration: Pasar parámetros entre notebooks

# 3. Testing: Probar diferentes escenarios fácilmente

# WIDGET 1: Text Input (entrada de texto libre)# 2. Dashboards interactivos: Usuarios filtran datos sin modificar código

dbutils.widgets.text(# 1. Jobs parametrizados: Mismo notebook, diferentes parámetros por ejecución

    "fecha_inicio",     # Nombre interno del widget (para leer valor)# 🔄 CASOS DE USO:

    "2026-05-01",       # Valor por defecto

    "📅 Fecha Inicio"  # Label visible al usuario# - multiselect(): Selección múltiple (filtros, columnas)

)# - combobox(): Dropdown con opción de texto custom (flexible)

# - dropdown(): Selección única (estados, tipos, categorías)

# WIDGET 2: Dropdown (selección única de lista cerrada)# - text(): Entrada de texto libre (fechas, nombres, paths)

dbutils.widgets.dropdown(# 💡 TIPOS DE WIDGETS DISPONIBLES:

    "region",           # Nombre interno

    "All",              # Valor por defecto seleccionadoprint("✅ Widgets creados")

    ["All", "North", "South", "East", "West"],  # Opciones disponibles

    "🌍 Región"       # Label)

)    "📊 Métricas"                      # Label

    ["amount", "quantity", "sale_id"], # Opciones disponibles

# WIDGET 3: Combobox (dropdown + texto libre)    "amount,quantity",                 # Valores por defecto (separados por coma)

dbutils.widgets.combobox(    "metricas",                        # Nombre interno

    "producto",         # Nombre internodbutils.widgets.multiselect(

    "All",              # Valor por defecto# WIDGET 4: Multiselect (selección múltiple)

    ["All"] + products, # Opciones predefinidas + permite texto custom

    "📦 Producto"      # Label)

In [ ]:
# Leer valores de widgets
fecha_inicio = dbutils.widgets.get("fecha_inicio")
region_filtro = dbutils.widgets.get("region")
producto_filtro = dbutils.widgets.get("producto")
metricas = dbutils.widgets.get("metricas").split(",")

print("📋 Parámetros seleccionados:")
print(f"  Fecha inicio: {fecha_inicio}")
print(f"  Región: {region_filtro}")
print(f"  Producto: {producto_filtro}")
print(f"  Métricas: {metricas}")

In [ ]:
# Aplicar filtros dinámicamente
df_filtered = df_sales.filter(col("date") >= fecha_inicio)

if region_filtro != "All":
    df_filtered = df_filtered.filter(col("region") == region_filtro)
    
if producto_filtro != "All":
    df_filtered = df_filtered.filter(col("product") == producto_filtro)

print(f"📊 Registros después de filtrar: {df_filtered.count()}")
display(df_filtered)

## Ejercicio 3: Debugging y Logging

In [ ]:
import time
from functools import wraps

def time_it(func):
    """Decorator para medir tiempo de ejecución"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"⏱️  {func.__name__} ejecutado en {end - start:.2f} segundos")
        return result
    return wrapper

@time_it
def procesar_datos(df):
    """Simula procesamiento de datos"""
    result = df.groupBy("region", "product").agg(
        sum("amount").alias("total"),
        avg("amount").alias("promedio")
    ).collect()
    return result

# Ejecutar
resultado = procesar_datos(df_sales)
print(f"✅ Procesado {len(resultado)} grupos")

In [ ]:
# Función para detectar columnas con nulos
def find_null_columns(df):
    """Encuentra columnas con valores nulos"""
    null_counts = []
    
    for col_name in df.columns:
        null_count = df.filter(col(col_name).isNull()).count()
        if null_count > 0:
            null_counts.append((col_name, null_count))
    
    return null_counts

# Probar
nulls = find_null_columns(df_sales)
if nulls:
    print("⚠️  Columnas con nulos:")
    for col_name, count in nulls:
        print(f"  - {col_name}: {count} nulos")
else:
    print("✅ No hay valores nulos")

In [ ]:
# Analizar plan de ejecución
df_complex = df_sales \
    .filter(col("amount") > 500) \
    .groupBy("region") \
    .agg(sum("amount").alias("total"))

print("📊 Plan de ejecución físico:")
df_complex.explain()

## Ejercicio 4: Guardar Resultados

In [ ]:
# Guardar resultados del análisis
table_name = "lab03_sales_analysis"

df_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("region") \
    .saveAsTable(table_name)

print(f"✅ Tabla creada: {table_name}")

In [ ]:
# Verificar particiones creadas
files = dbutils.fs.ls(output_path)
partitions = [f.name for f in files if f.name.startswith("region=")]

print(f"📁 Particiones creadas: {len(partitions)}")
for p in partitions:
    print(f"  - {p}")

## ✅ Lab Completado

Has aprendido:
- ✅ Magic commands (%md, %sql, %sh, %fs)
- ✅ Widgets para parametrización
- ✅ Debugging con decorators y timing
- ✅ Análisis de planes de ejecución
- ✅ Particionado de datos